# Task 1

### Imports

In [ ]:
from pathlib import Path
import re
from typing import Optional, Tuple, Dict, Any, List

import pandas as pd
from tqdm import tqdm
from ase.io import read
import numpy as np

### Paths and config

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parents[2]

DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = OUTPUT_DIR / "dataset_inventory.csv"
EXPECTED_CIF_COUNT = 2916

print("PROJECT_ROOT:", PROJECT_ROOT)    
print("DATA_ROOT:", DATA_ROOT)
print("OUT_CSV:", OUT_CSV)

### Helper functions

In [ ]:
def normalize_path(path: Path) -> str:
    return path.resolve().as_posix()


def relative_path(path: Path, root: Path) -> str:
    return path.resolve().relative_to(root.resolve()).as_posix()


def check_cif_readable(cif_path: Path) -> Tuple[bool, str]:
    try:
        atoms = read(str(cif_path))
        if atoms is None or len(atoms) == 0:
            return False, "empty_or_unparsed"
        return True, "ok"
    except Exception as e:
        return False, f"parse_error: {type(e).__name__}: {e}"


def parse_metadata_from_path(cif_path: Path, data_root: Path) -> Dict[str, Optional[float]]:
    """
    Expected pattern:
        data/r340/t9.6/t9.6_80.cif

    Parse:
        lower_rotation = 340     from folder r340
        displacement   = 9.6     from folder t9.6
        upper_rotation = 80      from filename t9.6_80.cif
    """
    rel = cif_path.resolve().relative_to(data_root.resolve())
    parts = rel.parts

    lower_rotation = None
    displacement = None
    upper_rotation = None

    # Expecting something like:
    # parts = ('r340', 't9.6', 't9.6_80.cif')
    if len(parts) >= 3:
        r_folder = parts[-3]   # e.g. r340
        t_folder = parts[-2]   # e.g. t9.6
        filename = Path(parts[-1]).stem  # e.g. t9.6_80

        # Parse lower rotation from r340
        m_r = re.fullmatch(r"r(-?\d+(?:\.\d+)?)", r_folder, flags=re.IGNORECASE)
        if m_r:
            lower_rotation = float(m_r.group(1))

        # Parse displacement from t9.6
        m_t = re.fullmatch(r"t(-?\d+(?:\.\d+)?)", t_folder, flags=re.IGNORECASE)
        if m_t:
            displacement = float(m_t.group(1))

        # Parse upper rotation from t9.6_80
        m_file = re.fullmatch(
            r"t(-?\d+(?:\.\d+)?)_(-?\d+(?:\.\d+)?)",
            filename,
            flags=re.IGNORECASE,
        )
        if m_file:
            # optional consistency check:
            file_disp = float(m_file.group(1))
            file_upper = float(m_file.group(2))

            # If folder displacement parsed already, keep it; otherwise take from filename
            if displacement is None:
                displacement = file_disp

            upper_rotation = file_upper

    return {
        "lower_rotation": lower_rotation,
        "displacement": displacement,
        "upper_rotation": upper_rotation,
    }


def build_structure_id(
    lower_rotation: Optional[float],
    displacement: Optional[float],
    upper_rotation: Optional[float],
    fallback_index: int,
) -> str:
    if (
        lower_rotation is not None
        and displacement is not None
        and upper_rotation is not None
    ):
        return f"L{lower_rotation:g}_D{displacement:g}_U{upper_rotation:g}"
    return f"STRUCT_{fallback_index:04d}"

### Scan CIF files

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_ROOT}")

cif_files = sorted(DATA_ROOT.rglob("*.cif"))
print("Total CIF files found:", len(cif_files))

### Build dataset inventory with tqdm

In [ ]:
rows: List[Dict[str, Any]] = []

for idx, cif_path in enumerate(tqdm(cif_files, desc="Scanning CIF files"), start=1):
    abs_path = normalize_path(cif_path)
    rel_path = relative_path(cif_path, DATA_ROOT)
    file_name = cif_path.name

    metadata = parse_metadata_from_path(cif_path, DATA_ROOT)
    is_readable, read_msg = check_cif_readable(cif_path)

    status = "ok" if is_readable else "parse_error"

    structure_id = build_structure_id(
        metadata["lower_rotation"],
        metadata["displacement"],
        metadata["upper_rotation"],
        idx,
    )

    rows.append(
        {
            "structure_id": structure_id,
            "cif_path": abs_path,
            "relative_cif_path": rel_path,
            "file_name": file_name,
            "lower_rotation": metadata["lower_rotation"],
            "displacement": metadata["displacement"],
            "upper_rotation": metadata["upper_rotation"],
            "status": status,
            "read_message": read_msg,
        }
    )

df_inventory = pd.DataFrame(rows)
df_inventory.head()

### Validation columns

In [ ]:
df_inventory["duplicate_file_name"] = df_inventory["file_name"].duplicated(keep=False)

df_inventory["duplicate_config"] = df_inventory.duplicated(
    subset=["lower_rotation", "displacement", "upper_rotation"],
    keep=False,
)

df_inventory["metadata_missing"] = df_inventory[
    ["lower_rotation", "displacement", "upper_rotation"]
].isnull().any(axis=1)

df_inventory = df_inventory.sort_values(
    by=["lower_rotation", "displacement", "upper_rotation", "file_name"],
    na_position="last",
).reset_index(drop=True)

df_inventory.insert(0, "structure_index", range(1, len(df_inventory) + 1))

df_inventory.head()

### Save CSV

In [ ]:
ordered_cols = [
    "structure_index",
    "structure_id",
    "cif_path",
    "relative_cif_path",
    "file_name",
    "lower_rotation",
    "displacement",
    "upper_rotation",
    "status",
    "read_message",
    "duplicate_file_name",
    "duplicate_config",
    "metadata_missing",
]

df_inventory = df_inventory[ordered_cols]
df_inventory.to_csv(OUT_CSV, index=False)

print(f"Saved dataset inventory to: {OUT_CSV}")

### Print summary

In [ ]:
total_found = len(df_inventory)
readable_count = (df_inventory["status"] == "ok").sum()
unreadable_count = (df_inventory["status"] != "ok").sum()
duplicate_name_count = df_inventory["duplicate_file_name"].sum()
duplicate_config_count = df_inventory["duplicate_config"].sum()
metadata_missing_count = df_inventory["metadata_missing"].sum()

print("=" * 60)
print("DATASET INVENTORY SUMMARY")
print("=" * 60)
print(f"Data root              : {DATA_ROOT}")
print(f"Total CIF files found  : {total_found}")
print(f"Expected CIF count     : {EXPECTED_CIF_COUNT}")
print(f"Count match            : {total_found == EXPECTED_CIF_COUNT}")
print(f"Readable CIFs          : {readable_count}")
print(f"Unreadable CIFs        : {unreadable_count}")
print(f"Duplicate file names   : {duplicate_name_count}")
print(f"Duplicate configs      : {duplicate_config_count}")
print(f"Missing parsed metadata: {metadata_missing_count}")
print(f"Saved inventory to     : {OUT_CSV}")
print("=" * 60)

# Task 2

### Paths

In [ ]:
INPUT_INVENTORY = PROJECT_ROOT / "outputs" / "dataset_inventory.csv"
INPUT_ENERGY = PROJECT_ROOT / "data" / "file_energy.csv"   

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_MASTER = OUTPUT_DIR / "dataset_master.csv"

print("Inventory file:", INPUT_INVENTORY)
print("Energy file   :", INPUT_ENERGY)
print("Output file   :", OUT_MASTER)

### Load Files

In [ ]:
df_inventory = pd.read_csv(INPUT_INVENTORY)
df_energy = pd.read_csv(INPUT_ENERGY)

print("Inventory shape:", df_inventory.shape)
print("Energy shape   :", df_energy.shape)

display(df_inventory.head())
display(df_energy.head())

### Clean and standardize energy table

In [ ]:
# Keep only useful columns
df_energy = df_energy.copy()

# Rename columns to project-standard names
df_energy = df_energy.rename(
    columns={
        "r_bottom": "lower_rotation",
        "r_upper": "upper_rotation",
        "energy_values_experimental": "energy",
        "adjusted_energy": "delta_energy",
    }
)

# Drop unnecessary unnamed index column if present
drop_cols = [col for col in ["Unnamed: 0"] if col in df_energy.columns]
if drop_cols:
    df_energy = df_energy.drop(columns=drop_cols)

# Standardize numeric dtypes
numeric_cols = ["lower_rotation", "displacement", "upper_rotation", "energy", "delta_energy"]
for col in numeric_cols:
    df_energy[col] = pd.to_numeric(df_energy[col], errors="coerce")

display(df_energy.head())
print(df_energy.dtypes)

### Use constant stable energy

In [ ]:
STABLE_ENERGY = -936.6398

df_energy["stable_energy"] = STABLE_ENERGY
df_energy["delta_energy"] = df_energy["energy"] - df_energy["stable_energy"]

print("Stable energy used:", STABLE_ENERGY)
display(df_energy.head())

### Clean and standardize energy table

In [ ]:
df_master = df_inventory.merge(
    df_energy[
        [
            "file_id",
            "lower_rotation",
            "displacement",
            "upper_rotation",
            "energy",
            "stable_energy",
            "delta_energy",
        ]
    ],
    on=["lower_rotation", "displacement", "upper_rotation"],
    how="left",
    validate="one_to_one",
)

print("Master shape:", df_master.shape)
display(df_master.head())

### Validate computed delta vs given adjusted energy

In [ ]:
df_master["missing_energy"] = df_master["energy"].isna()
df_master["missing_stable_energy"] = df_master["stable_energy"].isna()
df_master["missing_delta_energy"] = df_master["delta_energy"].isna()

print("Missing energy rows       :", int(df_master["missing_energy"].sum()))
print("Missing stable energy rows:", int(df_master["missing_stable_energy"].sum()))
print("Missing delta energy rows :", int(df_master["missing_delta_energy"].sum()))

In [ ]:
df_master["delta_energy_check"] = df_master["energy"] - df_master["stable_energy"]
df_master["delta_match"] = np.isclose(
    df_master["delta_energy"],
    df_master["delta_energy_check"],
    atol=1e-8,
)

print("Delta consistency mismatches:", int((~df_master["delta_match"]).sum()))

### Final cleaning cell

In [ ]:
final_cols = [
    "structure_id",
    "relative_cif_path",
    "lower_rotation",
    "displacement",
    "upper_rotation",
    "energy",
    "delta_energy",
]

df_master = df_master[final_cols]

display(df_master.head())

### Save Master CSV

In [ ]:
df_master.to_csv(OUT_MASTER, index=False)

print("Final dataset saved to:", OUT_MASTER)
print("Final shape:", df_master.shape)

# Task 3

### Paths

In [ ]:
INPUT_MASTER = PROJECT_ROOT / "outputs" / "dataset_master.csv"
OUT_METADATA = OUTPUT_DIR / "dataset_metadata.csv"

print("Input master :", INPUT_MASTER)
print("Output file  :", OUT_METADATA)

### Load Master file

In [ ]:
df_master = pd.read_csv(INPUT_MASTER)

print("dataset_master shape:", df_master.shape)
display(df_master.head())

### Helper functions

In [ ]:
def get_abs_cif_path(relative_cif_path: str, project_root: Path) -> Path:
    return project_root / "data" / relative_cif_path


def count_elements(symbols) -> Dict[str, int]:
    counts = pd.Series(symbols).value_counts().to_dict()
    return {
        "carbon_count": int(counts.get("C", 0)),
        "hydrogen_count": int(counts.get("H", 0)),
        "oxygen_count": int(counts.get("O", 0)),
    }


def extract_structure_metadata(cif_path: Path) -> Dict[str, Any]:
    """
    Read CIF and extract:
    - atom_count
    - lower_atom_count
    - upper_atom_count
    - C/H/O counts
    - parse_status

    Splitting logic:
    sort atoms by z-coordinate
    first half  -> lower layer
    second half -> upper layer
    """
    try:
        atoms = read(str(cif_path))

        if atoms is None or len(atoms) == 0:
            return {
                "atom_count": None,
                "lower_atom_count": None,
                "upper_atom_count": None,
                "carbon_count": None,
                "hydrogen_count": None,
                "oxygen_count": None,
                "parse_status": "empty_or_unparsed",
            }

        positions = atoms.get_positions()
        symbols = atoms.get_chemical_symbols()

        atom_count = len(atoms)

        # total composition counts
        elem_counts = count_elements(symbols)

        # z-sort split
        z_coords = positions[:, 2]
        z_order = z_coords.argsort()

        half = atom_count // 2
        lower_idx = z_order[:half]
        upper_idx = z_order[half:]

        lower_atom_count = len(lower_idx)
        upper_atom_count = len(upper_idx)

        parse_status = "ok"

        # optional validation-style status tagging
        if atom_count != 156:
            parse_status = "atom_count_error"
        elif lower_atom_count != 78 or upper_atom_count != 78:
            parse_status = "layer_split_error"

        return {
            "atom_count": int(atom_count),
            "lower_atom_count": int(lower_atom_count),
            "upper_atom_count": int(upper_atom_count),
            "carbon_count": int(elem_counts["carbon_count"]),
            "hydrogen_count": int(elem_counts["hydrogen_count"]),
            "oxygen_count": int(elem_counts["oxygen_count"]),
            "parse_status": parse_status,
        }

    except Exception:
        return {
            "atom_count": None,
            "lower_atom_count": None,
            "upper_atom_count": None,
            "carbon_count": None,
            "hydrogen_count": None,
            "oxygen_count": None,
            "parse_status": "parse_error",
        }

### Extract metadata from all CIFs

In [ ]:
metadata_rows: List[Dict[str, Any]] = []

for _, row in tqdm(df_master.iterrows(), total=len(df_master), desc="Building metadata"):
    rel_path = row["relative_cif_path"]
    cif_path = get_abs_cif_path(rel_path, PROJECT_ROOT)

    meta = extract_structure_metadata(cif_path)

    metadata_rows.append(
        {
            "structure_id": row["structure_id"],
            "relative_cif_path": rel_path,
            "lower_rotation": row["lower_rotation"],
            "displacement": row["displacement"],
            "upper_rotation": row["upper_rotation"],
            "energy": row["energy"],
            "delta_energy": row["delta_energy"],
            "atom_count": meta["atom_count"],
            "lower_atom_count": meta["lower_atom_count"],
            "upper_atom_count": meta["upper_atom_count"],
            "carbon_count": meta["carbon_count"],
            "hydrogen_count": meta["hydrogen_count"],
            "oxygen_count": meta["oxygen_count"],
            "parse_status": meta["parse_status"],
        }
    )

df_metadata = pd.DataFrame(metadata_rows)
display(df_metadata.head())

### Validation flags and saving csv

In [ ]:
df_metadata["atom_count_ok"] = df_metadata["atom_count"] == 156
df_metadata["lower_atom_count_ok"] = df_metadata["lower_atom_count"] == 78
df_metadata["upper_atom_count_ok"] = df_metadata["upper_atom_count"] == 78

df_metadata["carbon_count_ok"] = df_metadata["carbon_count"] == 48
df_metadata["hydrogen_count_ok"] = df_metadata["hydrogen_count"] == 72
df_metadata["oxygen_count_ok"] = df_metadata["oxygen_count"] == 36

display(df_metadata.head())

In [ ]:
final_cols = [
    "structure_id",
    "relative_cif_path",
    "lower_rotation",
    "displacement",
    "upper_rotation",
    "energy",
    "delta_energy",
    "atom_count",
    "lower_atom_count",
    "upper_atom_count",
    "carbon_count",
    "hydrogen_count",
    "oxygen_count"
]

df_metadata = df_metadata[final_cols]
display(df_metadata.head())

In [ ]:
df_metadata.to_csv(OUT_METADATA, index=False)

print(f"Saved dataset metadata to: {OUT_METADATA}")
print("Shape:", df_metadata.shape)

### Summary

In [ ]:
print("=" * 60)
print("DATASET METADATA SUMMARY")
print("=" * 60)
print(f"Total structures          : {len(df_metadata)}")
print(f"Saved file                : {OUT_METADATA}")
print("=" * 60)

# FINAL CODE

In [1]:
from pathlib import Path
import re

import pandas as pd
from tqdm import tqdm
from ase.io import read


# =========================
# CONFIG
# =========================
PROJECT_ROOT = Path.cwd().resolve().parents[2]
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CIF_COUNT = 2916
STABLE_ENERGY = -936.6398

INVENTORY_CSV = OUTPUT_DIR / "dataset_inventory.csv"
MASTER_CSV = OUTPUT_DIR / "dataset_master.csv"
ENERGY_CSV = DATA_ROOT / "file_energy.csv"


# =========================
# HELPERS
# =========================
def rel_to(path: Path, root: Path) -> str:
    return path.resolve().relative_to(root.resolve()).as_posix()


def read_cif_safe(cif_path: Path):
    try:
        atoms = read(str(cif_path))
        if atoms is None or len(atoms) == 0:
            return None, "empty_or_unparsed"
        return atoms, "ok"
    except Exception as e:
        return None, f"parse_error: {type(e).__name__}: {e}"


def parse_cif_metadata(cif_path: Path, data_root: Path) -> dict[str, float | None]:
    rel_parts = cif_path.resolve().relative_to(data_root.resolve()).parts
    lower_rotation = displacement = upper_rotation = None

    if len(rel_parts) >= 3:
        r_folder = rel_parts[-3]
        t_folder = rel_parts[-2]
        filename = Path(rel_parts[-1]).stem

        m_r = re.fullmatch(r"r(-?\d+(?:\.\d+)?)", r_folder, flags=re.IGNORECASE)
        m_t = re.fullmatch(r"t(-?\d+(?:\.\d+)?)", t_folder, flags=re.IGNORECASE)
        m_f = re.fullmatch(r"t(-?\d+(?:\.\d+)?)_(-?\d+(?:\.\d+)?)", filename, flags=re.IGNORECASE)

        if m_r:
            lower_rotation = float(m_r.group(1))
        if m_t:
            displacement = float(m_t.group(1))
        if m_f:
            if displacement is None:
                displacement = float(m_f.group(1))
            upper_rotation = float(m_f.group(2))

    return {
        "lower_rotation": lower_rotation,
        "displacement": displacement,
        "upper_rotation": upper_rotation,
    }


def make_structure_id(
    lower_rotation: float | None,
    displacement: float | None,
    upper_rotation: float | None,
    idx: int,
) -> str:
    if None not in (lower_rotation, displacement, upper_rotation):
        return f"L{lower_rotation:g}_D{displacement:g}_U{upper_rotation:g}"
    return f"STRUCT_{idx:04d}"


# =========================
# TASK 6: DATASET INVENTORY
# =========================
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_ROOT}")

cif_files = sorted(DATA_ROOT.rglob("*.cif"))

inventory_rows = []
for idx, cif_path in enumerate(tqdm(cif_files, desc="Task 6: scanning CIFs"), start=1):
    atoms, read_status = read_cif_safe(cif_path)
    meta = parse_cif_metadata(cif_path, DATA_ROOT)

    inventory_rows.append(
        {
            "structure_id": make_structure_id(
                meta["lower_rotation"], meta["displacement"], meta["upper_rotation"], idx
            ),
            "relative_cif_path": rel_to(cif_path, DATA_ROOT),
            "lower_rotation": meta["lower_rotation"],
            "displacement": meta["displacement"],
            "upper_rotation": meta["upper_rotation"]
        }
    )

df_inventory = pd.DataFrame(inventory_rows).sort_values(
    ["lower_rotation", "displacement", "upper_rotation"],
    na_position="last",
).reset_index(drop=True)

df_inventory.to_csv(INVENTORY_CSV, index=False)

print("=" * 60)
print("DATASET INVENTORY SUMMARY")
print("=" * 60)
print(f"Total CIF files found  : {len(df_inventory)}")
print(f"Expected CIF count     : {EXPECTED_CIF_COUNT}")
print(f"Count match            : {len(df_inventory) == EXPECTED_CIF_COUNT}")
print(f"Saved inventory to     : {INVENTORY_CSV}")
print("=" * 60)


# =========================
# TASK 7: DATASET MASTER
# =========================
df_energy = pd.read_csv(ENERGY_CSV).rename(
    columns={
        "r_bottom": "lower_rotation",
        "r_upper": "upper_rotation",
        "energy_values_experimental": "energy",
    }
)

if "Unnamed: 0" in df_energy.columns:
    df_energy = df_energy.drop(columns="Unnamed: 0")

for col in ["lower_rotation", "displacement", "upper_rotation", "energy"]:
    df_energy[col] = pd.to_numeric(df_energy[col], errors="coerce")

df_energy["delta_energy"] = df_energy["energy"] - STABLE_ENERGY

df_master = df_inventory.merge(
    df_energy[["lower_rotation", "displacement", "upper_rotation", "energy", "delta_energy"]],
    on=["lower_rotation", "displacement", "upper_rotation"],
    how="left",
    validate="one_to_one",
)

df_master = df_master[
    [
        "structure_id",
        "relative_cif_path",
        "lower_rotation",
        "displacement",
        "upper_rotation",
        "energy",
        "delta_energy",
    ]
]

df_master.to_csv(MASTER_CSV, index=False)

print("=" * 60)
print("DATASET MASTER SUMMARY")
print("=" * 60)
print(f"Rows                    : {len(df_master)}")
print(f"Missing energy rows     : {df_master['energy'].isna().sum()}")
print(f"Missing delta rows      : {df_master['delta_energy'].isna().sum()}")
print(f"Saved master to         : {MASTER_CSV}")
print("=" * 60)

Task 6: scanning CIFs:  15%|█▍        | 430/2916 [00:31<03:00, 13.78it/s]d:\masters_project\.venv\Lib\site-packages\ase\io\cif.py:410: UserWarning: crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!
  warnings.warn(
Task 6: scanning CIFs: 100%|██████████| 2916/2916 [04:56<00:00,  9.83it/s]

DATASET INVENTORY SUMMARY
Total CIF files found  : 2916
Expected CIF count     : 2916
Count match            : True
Saved inventory to     : D:\masters_project\outputs\dataset_inventory.csv
DATASET MASTER SUMMARY
Rows                    : 2916
Missing energy rows     : 0
Missing delta rows      : 0
Saved master to         : D:\masters_project\outputs\dataset_master.csv
